In [8]:
# Step 1: The Production Environment (No Hardcoded Keys)
# In Andrew Ng’s courses, you might have seen API keys pasted directly into Jupyter Notebooks. In a company, that is a massive security risk.

# Environment Variables: You store your API keys (like OPENAI_API_KEY) in a hidden .env file.

# Modern Packages: LangChain recently split its architecture. You don't just install langchain anymore; you install the core logic and the specific vendor packages.

!python3 -m pip install langchain-core langchain-groq python-dotenv

  Attempting uninstall: groq
    Found existing installation: groq 1.1.0
    Uninstalling groq-1.1.0:
      Successfully uninstalled groq-1.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-groq]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [13]:
# Step 2: The Core Pipeline (LCEL)The industry standard way to build with LangChain relies on LCEL. 
#         It uses the Python pipe operator (|) to chain components together, 
#         exactly like Unix command-line pipes.
#         A standard production pipeline always consists of three parts:
#         Prompt Template --> LLM --> Output Parser

import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

# 1. Load your API keys securely
load_dotenv()

# 2. Define the Prompt Template (The Input structure)
# This prevents users from having to write the system prompt every time.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a ruthless but helpful startup advisor. Analyze the given SaaS idea. Provide 1 pro, 1 con, and a 1-sentence verdict."),
    ("user", "Here is the idea: {startup_idea}")
])

# 3. Initialize the Model (The Engine)
# Temperature 0.0 makes the output deterministic and reliable for software.
model = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

# 4. Define the Output Parser (The Formatter)
# This strips away the weird metadata the API returns and just gives you the string text.
parser = StrOutputParser()

# 5. Build the Chain using LCEL
chain = prompt | model | parser

# 6. Execute the Chain
response = chain.invoke({"startup_idea": "A platform that directly connects niche content creators with brands for sponsorships."})

print(response)

**Pro:** The platform can capitalize on the growing demand for influencer marketing, which is expected to reach $24.1 billion by 2025, providing a significant revenue opportunity.

**Con:** The platform will face intense competition from established players like AspireIQ, Upfluence, and HYPR, which already offer similar services, making it challenging to differentiate and gain traction.

**Verdict:** This SaaS idea has the potential to disrupt the influencer marketing space, but it will require a unique value proposition and strategic partnerships to stand out in a crowded market.


In [ ]:
from pydantic import BaseModel, Field

# Define exactly what the data should look like
class StartupAnaly